In [3]:
import pandas as pd
from pathlib import Path
import sys

# Absolute project root
PROJECT_ROOT = Path(r"D:\Northwestern\MLDS414\Final_project\414-legal-document-intelligence")
sys.path.append(str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "train_clean.csv"

print("Data path:", DATA_PATH)
print("File exists:", DATA_PATH.exists())

# First check column names only
columns = pd.read_csv(DATA_PATH, nrows=0).columns.tolist()
print(columns)

Data path: D:\Northwestern\MLDS414\Final_project\414-legal-document-intelligence\data\processed\train_clean.csv
File exists: True
['id', 'case_name', 'court', 'date_filed', 'case_type', 'class_action_sought', 'summary/long', 'summary/short', 'summary/tiny', 'full_text', 'text_length']


In [5]:
import csv

# Increase CSV field size limit for very long legal documents
csv.field_size_limit(sys.maxsize)

usecols = [
    "id",
    "case_name",
    "case_type",
    "summary/long",
    "summary/short",
    "summary/tiny",
    "full_text",
    "text_length"
]

train_clean = pd.read_csv(
    DATA_PATH,
    usecols=usecols,
    dtype=str,
    engine="python",
    nrows=300
)

train_clean = train_clean.dropna(subset=["full_text"]).reset_index(drop=True)

# Rename columns for easier baseline notebook code
train_clean = train_clean.rename(columns={
    "id": "doc_id",
    "full_text": "document_text",
    "summary/short": "reference_summary"
})

print("Loaded shape:", train_clean.shape)
train_clean[["doc_id", "case_name", "case_type", "text_length"]].head()

Loaded shape: (300, 8)


,doc_id,case_name,case_type,text_length
0,EE-AL-0045,"EEOC v. House of Philadelphia Center, Inc.",Equal Employment,32174
1,PB-NJ-0003,Disability Rights New Jersey v. Velez,Public Benefits / Government Services,152724
2,EE-FL-0136,United States v. Palm Beach County,Equal Employment,41712
3,EE-CA-0305,Wynne v. McCormick & Schmick's Seafood Restara...,Equal Employment,367032
4,NH-NJ-0002,"U.S. v. Mercer County, New Jersey",Nursing Home Conditions,105806


In [6]:
from src.preprocess import chunk_text

In [9]:
sample_text = train_clean.iloc[0]["document_text"]

chunks = chunk_text(sample_text)
print("num chunks:", len(chunks))
print("first chunk preview:\n", chunks[0]["text"][:500])

num chunks: 15
first chunk preview:
 IN

THE

UNITED

STATES

DISTRICT

FILD COUR T

P19

.05

Nl

el

.s

FOR THE SOUTHERN DISTRICT OF ALABAMA

SOUTHERN DIVISION

EQUAL EMPLOYMENT OPPORTUNITY ]

COMMISSION, ]

] Plaintiff, ] Civil Action No. OSS- 0'53a -~

v.

]

]
COMPLAINT

] HOUSE OF PHILADELPHIA CENTER, INC . ]

JURY TRIAL DEMAND

Defendant .

]
]
] ]

NATURE OF THE ACTION This is an action under Title VII of the Civil Rights Act of 1964 and Title I of the Civil Rights Act of 1991 to correct unlawful employment practices on th


In [8]:
print(type(chunks))
print(type(chunks[0]))
print(chunks[0])

<class 'list'>
<class 'dict'>
{'chunk_id': 0, 'text': 'IN\n\nTHE\n\nUNITED\n\nSTATES\n\nDISTRICT\n\nFILD COUR T\n\nP19\n\n.05\n\nNl\n\nel\n\n.s\n\nFOR THE SOUTHERN DISTRICT OF ALABAMA\n\nSOUTHERN DIVISION\n\nEQUAL EMPLOYMENT OPPORTUNITY ]\n\nCOMMISSION, ]\n\n] Plaintiff, ] Civil Action No. OSS- 0\'53a -~\n\nv.\n\n]\n\n]\nCOMPLAINT\n\n] HOUSE OF PHILADELPHIA CENTER, INC . ]\n\nJURY TRIAL DEMAND\n\nDefendant .\n\n]\n]\n] ]\n\nNATURE OF THE ACTION This is an action under Title VII of the Civil Rights Act of 1964 and Title I of the Civil Rights Act of 1991 to correct unlawful employment practices on the basis of sex and to provide appropriate relief to Sharonda Griffin who was adversely affected by such practices . The Commission alleges that the Defendant discriminated against Sharonda Griffin because of her sex, female .\n\n1\n\n\x0cJURISDICTION AND VENU E 1 . Jurisdiction of this Court is invoked pursuant to 28 U .S .C. §§ 451, 1331, 1337, 1343 and 1345 . This action is authorized and i

In [16]:
from src.preprocess import chunk_text
from src.summarize import generate_summary

def safe_summarize_document(text, top_k=5, summary_type="long"):
    chunks = chunk_text(text)

    chunk_summaries = []

    for chunk in chunks[:top_k]:
        chunk_text_content = chunk["text"] if isinstance(chunk, dict) else chunk

        if len(chunk_text_content) < 200:
            continue

        try:
            summary = generate_summary(
                chunk_text_content,
                summary_type=summary_type
            )
        except Exception:
            summary = None

        # fallback because current generate_summary may return None for long/short
        if summary is None or str(summary).strip() == "":
            sentences = chunk_text_content.split(". ")
            summary = ". ".join(sentences[:3]).strip()

        if summary:
            chunk_summaries.append(str(summary).strip())

    return "\n\n".join(chunk_summaries)

In [17]:
sample_text = train_clean.iloc[0]["document_text"]

summary = safe_summarize_document(sample_text, top_k=5, summary_type="long")

print(summary[:1000])

IN

THE

UNITED

STATES

DISTRICT

FILD COUR T

P19

.05

Nl

el

.s

FOR THE SOUTHERN DISTRICT OF ALABAMA

SOUTHERN DIVISION

EQUAL EMPLOYMENT OPPORTUNITY ]

COMMISSION, ]

] Plaintiff, ] Civil Action No. OSS- 0'53a -~

v.

]

]
COMPLAINT

] HOUSE OF PHILADELPHIA CENTER, INC . ]

JURY TRIAL DEMAND

Defendant .

]
]
] ]

NATURE OF THE ACTION This is an action under Title VII of the Civil Rights Act of 1964 and Title I of the Civil Rights Act of 1991 to correct unlawful employment practices on the basis of sex and to provide appropriate relief to Sharonda Griffin who was adversely affected by such practices

days prior to the institution of this lawsuit, Sharonda Griffin filed a Charge of Discrimination with the Commission alleging violations of Title VII by Defendant Employer. All conditions precedent to the institution of this lawsuit have been fulfilled. 7

of life, in amounts to be determined at trial .
E. Order Defendant Employer to pay Sharonda Griffin punitive damages for its mal

In [18]:
from src.evaluate import evaluate_summary

sample = train_clean.iloc[0]

pred = safe_summarize_document(
    sample["document_text"],
    top_k=5,
    summary_type="long"
)

scores = evaluate_summary(
    pred,
    sample["summary/long"]
)

print(pred[:1000])
scores

IN

THE

UNITED

STATES

DISTRICT

FILD COUR T

P19

.05

Nl

el

.s

FOR THE SOUTHERN DISTRICT OF ALABAMA

SOUTHERN DIVISION

EQUAL EMPLOYMENT OPPORTUNITY ]

COMMISSION, ]

] Plaintiff, ] Civil Action No. OSS- 0'53a -~

v.

]

]
COMPLAINT

] HOUSE OF PHILADELPHIA CENTER, INC . ]

JURY TRIAL DEMAND

Defendant .

]
]
] ]

NATURE OF THE ACTION This is an action under Title VII of the Civil Rights Act of 1964 and Title I of the Civil Rights Act of 1991 to correct unlawful employment practices on the basis of sex and to provide appropriate relief to Sharonda Griffin who was adversely affected by such practices

days prior to the institution of this lawsuit, Sharonda Griffin filed a Charge of Discrimination with the Commission alleging violations of Title VII by Defendant Employer. All conditions precedent to the institution of this lawsuit have been fulfilled. 7

of life, in amounts to be determined at trial .
E. Order Defendant Employer to pay Sharonda Griffin punitive damages for its mal

{'rouge1_f1': 0.3423076923076923,
 'rouge2_f1': 0.11969111969111969,
 'rougeL_f1': 0.19615384615384615}